<div align="center">

<a href="https://colab.research.google.com/github/utkukose/rd-project-management-NB-lecture/blob/main/weeks/week-07/NB07_proposal_writing_2.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 07: Proposal Writing II: Evaluation, Reviewers and Budget Justification

**R&D and Project Management in Computer Science (11117BLG002)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/rd-project-management-NB-lecture/weeks/week-07/lab.html>.

## Overview

Proposals are judged by people under time pressure, and their judgements vary. This week examines how proposals are evaluated, what research shows about the reliability of peer review, how to write for reviewers and how to justify a budget. A mock panel lets students take the reviewer's seat [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to describe the stages of proposal evaluation, to explain the evidence on reviewer disagreement and its consequences near the funding line, to identify common weaknesses of proposals, and to justify a budget in relation to tasks.

## How proposals are evaluated

Evaluation usually proceeds in stages. Experts first review proposals individually against the criteria of the call, then reach consensus in a group or panel, and the funder ranks proposals and funds as many as the budget allows. In Horizon Europe, independent experts score each proposal on excellence, impact and implementation, with thresholds and ranking rules that Week 5 introduced [1]. TÜBİTAK's research programmes rely on panels and reviewers who assess originality, method, project management and wider impact with equal weight [4]. The ERC uses a single criterion, scientific excellence, applied to both the project and the principal investigator.

## How reliable is peer review?

Peer review of proposals is less reliable than applicants often assume. Pier and colleagues asked experienced reviewers to evaluate the same grant applications of the National Institutes of Health and found low agreement between reviewers, both in their scores and in their written critiques [2]. Graves, Barnett and Clarke analysed scores by members of a grant review panel and showed that the variation between reviewers could change funding decisions for proposals near the funding line [3]. Figure 7.1 illustrates the mechanism with a simple model: When reviewers' scores contain noise, the better of two similar proposals is ranked higher only with a modest probability, and adding reviewers helps but does not remove the effect. The lab simulates whole panels and shows how much the set of funded proposals can change between two panels that see the same proposals.

![Figure 7.1](https://raw.githubusercontent.com/utkukose/rd-project-management-NB-lecture/main/weeks/week-07/figures/w07_fig1.png)

*Figure 7.1. Probability that the better of two proposals whose true quality differs by half a point is ranked higher, as a function of reviewer noise and the number of reviewers per proposal, in a simple normal model.*

## Writing for reviewers

The practical conclusion is to reduce the room for misreading. Reviewers look for the answer to each criterion, so the proposal should make it easy to find: headings that follow the criteria, objectives stated once and referred to consistently, a method that addresses every objective, preliminary results that make feasibility credible and a work plan whose effort matches the ambition. Common weaknesses include vague objectives, a method that does not match the objectives, missing risk analysis, impact claims without users or measures, and budgets that are not linked to tasks. The lab asks students to sort such reviewer comments by criterion, which trains the habit of reading one's own draft as a reviewer would.

## Budget justification

A budget is part of the argument. Each cost should follow from a task: person-months from the effort of work packages, equipment from a method that needs it and cannot use existing infrastructure, travel from dissemination and collaboration, services from tasks that the team cannot do itself. Funders define eligible cost categories and limits in their rules, and Horizon Europe adds a flat rate for indirect costs to eligible direct costs [5]. Reviewers are sensitive to both overbudgeting, which signals poor planning, and underbudgeting, which signals that the work cannot be done as described.

> **Pause and reflect.** In the mock panel, how did your scores compare with the scores you would expect for your own proposal? What does the difference suggest?

# Hands-on lab

The notebook models reviewer noise, computes the probability that the better of two proposals is ranked higher and simulates the agreement between two independent panels [2, 3].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Two proposals, noisy reviewers

Each reviewer reports the true quality plus normal noise. With k reviewers per proposal, the difference between the mean scores of two proposals is normal with standard deviation sd * sqrt(2 / k).

### Your turn, exercise 1

Compute the probability that the better proposal is ranked higher when the true difference is 0.5, the reviewer noise is 1.0 and each proposal has 3 reviewers.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
from scipy.stats import norm
def _pbetter_reference():
    return float(norm.cdf(0.5 / (1.0 * np.sqrt(2 / 3))))

In [ ]:
p_better = None  # your computation
check("Exercise 1", p_better, _pbetter_reference())

## 2. Two panels, one call

In [ ]:
def panel_overlap(sd=0.8, k=3, funded=5, n=20, trials=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    quality = np.linspace(2.5, 4.5, n)
    shares = []
    for _ in range(trials):
        a = (quality + rng.normal(0, sd, (k, n)).mean(0)).argsort()[-funded:]
        b = (quality + rng.normal(0, sd, (k, n)).mean(0)).argsort()[-funded:]
        shares.append(len(set(a) & set(b)) / funded)
    return float(np.mean(shares))

for sd in [0.2, 0.5, 0.8, 1.2]:
    print(f"noise {sd}: mean overlap {panel_overlap(sd=sd):.2f}")

### Your turn, exercise 2

Compute the mean overlap for noise 0.8 with 5 reviewers per proposal, keeping the other defaults, and compare it with 3 reviewers. Write two sentences on what the result implies for applicants whose proposals score just below the funding line.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _overlap_reference():
    return panel_overlap(sd=0.8, k=5)

In [ ]:
overlap_five = None  # your computation
check("Exercise 2", overlap_five, _overlap_reference())

## Visual exploration: Noisy reviewers and a small true difference

Two proposals differ slightly in true quality. The curves show the chance that the better one receives the higher mean score, for panels of different sizes and reviewers with different noise.

In [ ]:
rng = np.random.default_rng(SEED)
delta, panel = 0.3, [1, 2, 3, 5, 8, 12]
for sd in [0.5, 1.0, 1.5]:
    p = [np.mean(rng.normal(delta, sd, (5000, k)).mean(1) > rng.normal(0, sd, (5000, k)).mean(1)) for k in panel]
    plt.plot(panel, p, "o-", label=f"reviewer noise sd = {sd}")
plt.axhline(0.5, color="k", ls="--", lw=0.8)
plt.xlabel("reviewers per proposal"); plt.ylabel("chance that the better proposal ranks higher"); plt.title("A small true difference and noisy reviewers"); plt.legend(); plt.show()

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/rd-project-management-NB-lecture/weeks/week-07/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which of the three summaries in the mock panel did you rank first, and which comment would you write to its authors?
2. Given the evidence on reviewer disagreement, how should an applicant respond to a rejection with good scores?
3. Which item of your draft budget would a reviewer question first, and how will you justify it?

## Weekly task and submission

Review one anonymised proposal draft from another student, or a public example, using the official criteria of the target programme: give a score and a comment of about 150 words per criterion. Then write the budget justification for your own proposal, linking every item to tasks. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**How reliable is grant peer review?.** Review the evidence on the reliability and bias of grant peer review and on proposed remedies such as more reviewers, structured criteria and partial lotteries [2, 3].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## Midterm capstone

This week closes the first half of the course with the [Midterm Capstone: A Research Proposal](https://github.com/utkukose/rd-project-management-NB-lecture/blob/main/exams/midterm/README.md).

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the midterm capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 7, with the report and all code files attached or linked.

## References

[1] European Commission (2021). *Standard briefing slides for experts: Horizon Europe*. <https://ec.europa.eu/info/funding-tenders/opportunities/docs/2021-2027/experts/standard-briefing-slides-for-experts_he_en.pdf>

[2] Pier, E. L., Brauer, M., Filut, A., Kaatz, A., Raclaw, J., Nathan, M. J., Ford, C. E., & Carnes, M. (2018). Low agreement among reviewers evaluating the same NIH grant applications. *Proceedings of the National Academy of Sciences*, 115(12), 2952-2957. <https://doi.org/10.1073/pnas.1714379115>

[3] Graves, N., Barnett, A. G., & Clarke, P. (2011). Funding grant proposals for scientific research: Retrospective analysis of scores by members of grant review panel. *BMJ*, 343, d4797. <https://doi.org/10.1136/bmj.d4797>

[4] TÜBİTAK (2020). *ARDEB 1001 Programı: Proje değerlendirme sistemindeki yenilikler*. <https://tubitak.gov.tr/tr/duyuru/ardeb-1001-programi-proje-degerlendirme-sistemindeki-yenilikler>

[5] European Commission (2024). *Lump sum funding in Horizon Europe: What do I need to know?*. <https://ec.europa.eu/info/funding-tenders/opportunities/docs/2021-2027/horizon/guidance/ls-funding-what-do-i-need-to-know_he_en.pdf>